# DOE Jupyter Runtime ML Graphs

Run this notebook on the DOE desktop after cloning or downloading the `north-slope-gas-hydrates` repo. It runs the ML/header/equation graph workflow while keeping approved workbook rows, predictions, models, and local configs out of GitHub.

## Guardrails

- Do not paste approved workbook rows into this notebook.
- Do not commit `outputs_runtime/`, `models_runtime/`, or populated local configs.
- Saturation and occurrence fields are Y-only labels, not predictors.
- Stability is context/admissibility only, not hydrate proof.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    markers = [
        'PROJECT_CONTEXT.md',
        '01_pipeline/run_three_dataset_ml_pipeline.py',
        'code_transfer_block/multi_saturation_target_workflow.py',
    ]
    for candidate in [start, *start.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise FileNotFoundError('Run this notebook from inside the north-slope-gas-hydrates repo.')

REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
print(REPO_ROOT)

## Configure Local DOE Paths

Option A: edit `DATA_DIR` below. Option B: copy `runtime_config.template.json` to `configs_local/doe_jupyter_runtime_config.json` and edit that local ignored config.

In [ ]:
CONFIG_PATH = REPO_ROOT / 'configs_local' / 'doe_jupyter_runtime_config.json'

if CONFIG_PATH.exists():
    config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
else:
    config = {
        'data_dir': str(Path.home() / 'Downloads' / 'Northslopedatasets06052026'),
        'target': 'auto',
        'target_task': 'auto',
        'model': 'baseline',
        'run_single_target_pipeline': False,
        'run_slide_paper_visuals': True,
        'equation_input': '',
        'equation_sheet': None,
    }

DATA_DIR = Path(config['data_dir']).expanduser()
print('DATA_DIR:', DATA_DIR)
for name in ['curated_dataset1.xlsx', 'curated_dataset2.xlsx', 'curated_dataset3.xlsx']:
    print(name, 'FOUND' if (DATA_DIR / name).exists() else 'MISSING')

## Run The Public-Safe Export Pack

This calls the repo scripts and writes outputs under ignored `outputs_runtime/doe_jupyter_pack_*` folders.

In [ ]:
cmd = [
    sys.executable,
    'doe_jupyter_runtime_pack/run_public_safe_ml_graph_exports.py',
    '--data-dir', str(DATA_DIR),
]

if CONFIG_PATH.exists():
    cmd.extend(['--config', str(CONFIG_PATH)])

print(' '.join(cmd))
completed = subprocess.run(cmd, cwd=REPO_ROOT, text=True, capture_output=True)
print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
print('return code:', completed.returncode)

## Review Local Outputs

Open the newest `outputs_runtime/doe_jupyter_pack_*` folder. These are local review outputs, not GitHub artifacts unless separately sanitized and approved.

In [ ]:
runtime_dirs = sorted((REPO_ROOT / 'outputs_runtime').glob('doe_jupyter_pack_*'), key=lambda p: p.stat().st_mtime, reverse=True)
latest = runtime_dirs[0] if runtime_dirs else None
print('latest:', latest)
if latest:
    for path in sorted(latest.rglob('*')):
        if path.is_file():
            print(path.relative_to(REPO_ROOT))

## Display Slide/Word PNGs If Available

In [ ]:
from IPython.display import Image, display

candidate_pngs = []
if latest:
    candidate_pngs.extend(latest.rglob('*.png'))
candidate_pngs.extend((REPO_ROOT / 'outputs_runtime' / 'figures').glob('*.png'))

for png in list(dict.fromkeys(candidate_pngs))[:12]:
    print(png.relative_to(REPO_ROOT))
    display(Image(filename=str(png)))

## Optional Local-Only Well-Log Or Core Plots

Use this only with DOE-approved local tables. Do not push generated well-log panels or core-log plots unless they are anonymized, row-free/sanitized, and approved.

In [ ]:
# Example only. Uncomment and edit for a local approved table.
# import pandas as pd
# from dashboard.runtime.plotting import build_log_panel, build_core_log_crossplot
# logs = pd.read_csv('PATH_TO_LOCAL_APPROVED_LOG_TABLE.csv')
# fig = build_log_panel(logs, well_alias='ANON_WELL_1', columns=('gr_api', 'rt_ohm_m', 'rhob_g_cc'))
# fig.show()
# fig.write_image(str(REPO_ROOT / 'outputs_runtime' / 'local_only_well_log_panel.png'))